# Lezione 10. Rainbow e l'apprendimento distribuzionale: quando più miglioramenti si sommano

**DIH DATalife — Corso di Reinforcement Learning (RL) da zero**

Nella lezione sette hai conosciuto Deep Q-Network (DQN): un agente che combina l'apprendimento-Q con una rete neurale profonda, una rete target e il replay dell'esperienza per imparare direttamente dall'esperienza. È stata una tappa importante, ma non il punto finale. Da quando è stato pubblicato, la comunità di ricerca sull'apprendimento per rinforzo (RL) ha proposto miglioramenti concreti a diverse parti dello stesso algoritmo: come sceglie le azioni, cosa ricorda, come esplora, cosa prevede. Oggi vedi cosa succede quando le metti tutte insieme: il risultato si chiama **Rainbow**.

## Scaletta della lezione

1. Rainbow: la somma di sei miglioramenti indipendenti
2. Cosa ha rivelato lo studio di ablazione
3. Linee di ricerca attive che vedrai citate in letteratura
4. Codice dal vivo: aggiungi un'architettura dueling al tuo DQN
5. Punti chiave e cosa ti aspetta nella lezione 11


## 1. Rainbow: la somma di sei miglioramenti indipendenti

*Traccia della lezione: disegna un arcobaleno alla lavagna con sei strisce e scrivi il nome di ogni miglioramento su una striscia man mano che lo presenti.*

Un arcobaleno nasce dalla combinazione di colori che già esistevano in modo indipendente. Allo stesso modo, Rainbow nasce quando si uniscono sei miglioramenti che erano già stati provati con successo, ciascuno per conto proprio, sul DQN originale. Nessun componente è esclusivo di Rainbow: ciò che aggiunge l'insieme è la prova che integrarli tutti nello stesso agente funziona meglio di qualsiasi singolo miglioramento preso da solo.

Ripassali uno per uno:

- **Doppio apprendimento-Q**: separa la scelta della migliore azione (rete online) dalla sua valutazione (rete target), per ridurre la sovrastima tipica di DQN
- **Replay dell'esperienza prioritizzato (PER)**: invece di campionare le transizioni a caso, ripete più spesso quelle in cui l'agente ha sbagliato, perché sono quelle che portano più informazione nuova
- **Architettura dueling**: separa la previsione in due domande: quanto vale lo stato in generale —V(s)— e, all'interno di quello stato, quanto è buona ciascuna azione rispetto alle altre —A(s,a)—, invece di prevedere un unico valore Q per ogni azione
- **Ritorni multi-passo**: accumula la ricompensa di diversi passi consecutivi prima di fare bootstrap con la rete target, invece di guardare solo un passo in avanti
- **Apprendimento distribuzionale (C51)**: impara l'intera distribuzione dei ritorni possibili, non solo la loro media, rappresentata con 51 punti fissi (atomi) e confrontata con la distribuzione obiettivo tramite la divergenza di Kullback-Leibler (KL)
- **Reti rumorose (Noisy Nets)**: sostituiscono l'esplorazione epsilon-greedy con un rumore apprendibile sui pesi della rete, così è la rete stessa a decidere quanta esplorazione serve in ogni situazione

Due precisazioni prima di proseguire. Primo: la distribuzione che impara C51 misura la variabilità reale dei possibili risultati, non l'incertezza dell'agente. Secondo: l'esempio visivo che il materiale di riferimento usa per l'architettura dueling è un videogioco di corse, dove la testa di valore guarda la scena complessiva (sto andando bene nella gara?) mentre la testa di vantaggio si concentra sui dettagli immediati, come l'auto che hai accanto.


## 2. Cosa ha rivelato lo studio di ablazione

*Traccia della lezione: chiedi prima di mostrare la risposta — "quale miglioramento pensate che apporti di meno una volta combinato con gli altri?"*

Quando il team che ha proposto Rainbow ha provato a togliere ciascun componente uno alla volta, il risultato non è stato simmetrico:

- L'apprendimento multi-passo, il replay prioritizzato e l'apprendimento distribuzionale sono risultati quelli che più contribuiscono alla prestazione finale
- Il doppio apprendimento-Q e l'architettura dueling hanno aggiunto meno del previsto una volta combinati con il resto: una spiegazione proposta è che l'apprendimento distribuzionale limita già di per sé l'intervallo dei ritorni possibili, riducendo gran parte del problema di sovrastima che il doppio apprendimento-Q era stato progettato per risolvere

In termini di efficienza sui dati, la cifra che si cita di solito è impressionante: Rainbow raggiunge, in circa sette milioni di fotogrammi di addestramento, lo stesso punteggio medio normalizzato che il DQN standard richiede circa 200 milioni di fotogrammi per ottenere. È un miglioramento di efficienza sui dati di circa 14 volte, con un tempo di calcolo simile a quello del DQN originale.


## 3. Linee di ricerca attive che vedrai citate in letteratura

Al di là di Rainbow, troverai altri nomi che circolano spesso nell'RL avanzato. Il materiale di riferimento di questa lezione li menziona solo di sfuggita, quindi trattali come una mappa di ciò che esiste, non come contenuto sviluppato qui:

- **RL multi-agente**: più agenti che imparano contemporaneamente nello stesso ambiente, a volte cooperando e a volte competendo; concetti ricorrenti includono l'addestramento centralizzato con esecuzione decentralizzata e la modellazione del comportamento degli altri agenti
- **RL gerarchico**: l'agente impara anche azioni estese nel tempo, chiamate opzioni, ciascuna con una propria politica interna e una propria condizione di terminazione, invece di imparare solo movimenti di un passo
- **Meta-apprendimento, apprendimento per trasferimento e apprendimento curricolare**: modi in cui un agente sfrutta ciò che ha imparato in un compito per risolvere più in fretta un compito correlato, oppure organizza l'addestramento in un ordine di difficoltà crescente
- **Sicurezza nell'RL**: un'area che acquista rilevanza man mano che questi agenti si avvicinano ad ambienti reali; l'apprendimento distribuzionale, per esempio, già offre un contributo utile qui, perché informa sulla probabilità di terminazioni premature o indesiderate, e non solo sul ritorno atteso

**Se qualcuno di questi nomi ti interessa, annotalo come linea di studio personale.**


## 4. Codice dal vivo: aggiungi un'architettura dueling al tuo DQN

*Traccia della lezione: prima di toccare il codice, richiama ad alta voce i componenti del DQN della lezione sette: rete online, rete target, buffer di replay ed esplorazione epsilon-greedy. Oggi cambi solo un componente: l'architettura interna della rete.*

Implementare i sei miglioramenti di Rainbow in una sola lezione non sarebbe realistico. Riprenderai invece il DQN minimo della lezione sette e gli aggiungerai un solo miglioramento, ben delimitato: l'architettura dueling. Addestrerai due agenti su CartPole (un DQN standard e un Dueling DQN) con esattamente lo stesso ciclo di addestramento, e confronterai le loro curve di ricompensa. L'unica differenza tra i due sarà l'architettura interna della rete.

**Esegui ogni cella in ordine.** Sono divise in passi piccoli, così puoi fermarti, commentare il risultato e rispondere alle domande prima di proseguire.


In [ ]:
!pip install gymnasium


### Prepariamo l'ambiente di lavoro

Importiamo gymnasium (la libreria standard di ambienti per l'RL), PyTorch per la rete neurale, e fissiamo un seme perché l'esperimento sia riproducibile.


In [ ]:
import random
from collections import deque, namedtuple

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Fijamos semillas para que el experimento sea reproducible
SEMILLA = 0
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)


### Il buffer di replay dell'esperienza

*Traccia della lezione: ricorda che questo componente lo hai già costruito nella lezione sette — oggi non cambia.*

`BufferReplay` memorizza le transizioni passate e permette di campionare lotti a caso, rompendo la correlazione temporale dei dati consecutivi di un episodio. È identico a quello della lezione sette.


In [ ]:
Transicion = namedtuple(
    "Transicion", ["estado", "accion", "recompensa", "siguiente_estado", "terminado"]
)


class BufferReplay:
    """Memoria de repeticion de experiencia (clase siete): guarda transiciones
    pasadas y permite muestrear lotes al azar, rompiendo la correlacion
    temporal de los datos consecutivos de un episodio."""

    def __init__(self, capacidad=10000):
        self.memoria = deque(maxlen=capacidad)

    def guardar(self, *args):
        self.memoria.append(Transicion(*args))

    def muestrear(self, tamano_lote):
        return random.sample(self.memoria, tamano_lote)

    def __len__(self):
        return len(self.memoria)


### La rete del DQN standard

Una sola testa che produce direttamente un valore Q per ogni azione possibile. È l'architettura "da manuale" che hai già visto nella lezione sette.


In [ ]:
class RedQEstandar(nn.Module):
    """DQN "de manual": una sola cabeza que produce directamente
    un valor Q por cada accion posible."""

    def __init__(self, n_entradas, n_acciones, n_ocultas=128):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(n_entradas, n_ocultas),
            nn.ReLU(),
            nn.Linear(n_ocultas, n_ocultas),
            nn.ReLU(),
            nn.Linear(n_ocultas, n_acciones),
        )

    def forward(self, x):
        return self.red(x)


### La rete del Dueling DQN

*Traccia della lezione: disegna alla lavagna il tronco condiviso e le due teste prima di mostrare il codice. Insisti sulla formula di combinazione: perché sottraiamo la media del vantaggio.*

Un tronco condiviso alimenta due teste: una stima V(s) (quanto vale lo stato, un unico numero) e l'altra stima A(s,a) (quanto è migliore ciascuna azione rispetto alle altre). Si combinano sottraendo la media del vantaggio, così che la somma resti su una scala stabile:

**Q(s,a) = V(s) + (A(s,a) − media_a[A(s,a)])**


In [ ]:
class RedQDueling(nn.Module):
    """Arquitectura dueling: un tronco compartido alimenta dos cabezas.
    Una estima V(s) -- cuanto vale el estado, un unico numero --
    y otra estima A(s,a) -- cuanto mejor es cada accion frente a las demas.
    Se combinan restando la media de la ventaja, para que la suma
    quede en una escala estable."""

    def __init__(self, n_entradas, n_acciones, n_ocultas=128):
        super().__init__()
        self.tronco = nn.Sequential(
            nn.Linear(n_entradas, n_ocultas),
            nn.ReLU(),
        )
        self.cabeza_valor = nn.Sequential(
            nn.Linear(n_ocultas, n_ocultas),
            nn.ReLU(),
            nn.Linear(n_ocultas, 1),
        )
        self.cabeza_ventaja = nn.Sequential(
            nn.Linear(n_ocultas, n_ocultas),
            nn.ReLU(),
            nn.Linear(n_ocultas, n_acciones),
        )

    def forward(self, x):
        caracteristicas = self.tronco(x)
        valor = self.cabeza_valor(caracteristicas)
        ventaja = self.cabeza_ventaja(caracteristicas)
        # Q(s,a) = V(s) + (A(s,a) - media_a[A(s,a)])
        # dim=-1 funciona igual con un estado suelto (sin lote) que con un
        # lote de estados: siempre promedia sobre el eje de las acciones.
        return valor + (ventaja - ventaja.mean(dim=-1, keepdim=True))


### Il ciclo di addestramento

*Traccia della lezione: sottolinea che il ciclo è esattamente lo stesso per le due reti. L'unica differenza tra i due esperimenti è quale classe di rete passi a `entrenar`.*

`entrenar` riceve la classe della rete come parametro. Il resto dell'algoritmo —rete online, rete target, buffer di replay, esplorazione epsilon-greedy decrescente— non cambia tra un esperimento e l'altro.


In [ ]:
def entrenar(clase_red, nombre, n_episodios=120, gamma=0.99, tamano_lote=64,
             tasa_aprendizaje=1e-3, actualizar_objetivo_cada=10):
    """Bucle de entrenamiento DQN estandar: red online, red objetivo,
    buffer de repeticion y exploracion epsilon-greedy decreciente.
    Cambiar `clase_red` entre RedQEstandar y RedQDueling es la unica
    diferencia entre ambos experimentos -- el resto del algoritmo no cambia."""

    entorno = gym.make("CartPole-v1")
    n_entradas = entorno.observation_space.shape[0]
    n_acciones = entorno.action_space.n

    red_online = clase_red(n_entradas, n_acciones)
    red_objetivo = clase_red(n_entradas, n_acciones)
    red_objetivo.load_state_dict(red_online.state_dict())
    red_objetivo.eval()

    optimizador = optim.Adam(red_online.parameters(), lr=tasa_aprendizaje)
    buffer = BufferReplay(capacidad=10000)

    epsilon, epsilon_minimo, decaimiento = 1.0, 0.05, 0.97
    recompensas_por_episodio = []

    for episodio in range(n_episodios):
        estado, _ = entorno.reset(seed=SEMILLA + episodio)
        recompensa_total = 0.0
        terminado = False

        while not terminado:
            if random.random() < epsilon:
                accion = entorno.action_space.sample()  # exploracion
            else:
                with torch.no_grad():
                    q_valores = red_online(torch.tensor(estado, dtype=torch.float32))
                    accion = int(torch.argmax(q_valores).item())  # explotacion

            siguiente_estado, recompensa, fin, truncado, _ = entorno.step(accion)
            terminado = fin or truncado
            buffer.guardar(estado, accion, recompensa, siguiente_estado, fin)
            estado = siguiente_estado
            recompensa_total += recompensa

            # Solo entrenamos cuando el buffer tiene datos suficientes
            if len(buffer) >= tamano_lote:
                lote = buffer.muestrear(tamano_lote)
                estados = torch.tensor(np.array([t.estado for t in lote]), dtype=torch.float32)
                acciones = torch.tensor([t.accion for t in lote], dtype=torch.int64)
                recompensas = torch.tensor([t.recompensa for t in lote], dtype=torch.float32)
                siguientes = torch.tensor(np.array([t.siguiente_estado for t in lote]), dtype=torch.float32)
                terminados = torch.tensor([t.terminado for t in lote], dtype=torch.float32)

                q_actuales = red_online(estados).gather(1, acciones.unsqueeze(1)).squeeze(1)
                with torch.no_grad():
                    q_siguientes = red_objetivo(siguientes).max(dim=1)[0]
                    objetivo = recompensas + gamma * q_siguientes * (1 - terminados)

                perdida = nn.functional.mse_loss(q_actuales, objetivo)
                optimizador.zero_grad()
                perdida.backward()
                optimizador.step()

        recompensas_por_episodio.append(recompensa_total)
        epsilon = max(epsilon_minimo, epsilon * decaimiento)

        if episodio % actualizar_objetivo_cada == 0:
            red_objetivo.load_state_dict(red_online.state_dict())

    entorno.close()
    print(f"{nombre}: recompensa media de los ultimos 20 episodios = "
          f"{np.mean(recompensas_por_episodio[-20:]):.1f}")
    return recompensas_por_episodio


### Addestriamo entrambi gli agenti

**Esegui questa cella e sfrutta il tempo di attesa (dura tra uno e due minuti su Colab) per ripassare ad alta voce i quattro elementi del ciclo.**


In [ ]:
# --- Entrenamos ambos agentes con el mismo bucle ---
recompensas_dqn = entrenar(RedQEstandar, "DQN estandar")
recompensas_dueling = entrenar(RedQDueling, "Dueling DQN")


### Confrontiamo le curve di ricompensa

Lisciamo ogni curva con una media mobile di dieci episodi, così da vedere la tendenza senza il rumore da episodio a episodio.


In [ ]:
def media_movil(valores, ventana=10):
    return np.convolve(valores, np.ones(ventana) / ventana, mode="valid")


# --- Comparamos las curvas de recompensa ---
plt.figure(figsize=(10, 5))
plt.plot(media_movil(recompensas_dqn), label="DQN estandar")
plt.plot(media_movil(recompensas_dueling), label="Dueling DQN")
plt.xlabel("Episodio")
plt.ylabel("Recompensa (media movil de 10 episodios)")
plt.title("CartPole-v1: DQN estandar vs. Dueling DQN")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


**Lettura del risultato**

*Traccia della lezione: fermati su questo grafico e chiedi alla classe quale curva si aspettava e quale vede. Non forzare una conclusione: su CartPole il vantaggio del Dueling DQN di solito è modesto, e quale rete vince può variare tra un'esecuzione e l'altra.*

Osserva due cose: quanto tempo impiega ciascuna curva ad avvicinarsi alla ricompensa massima di CartPole (500 passi) e quanta variabilità mostra ciascuna tra un episodio e l'altro. In un ambiente semplice come CartPole, il vantaggio del Dueling DQN di solito è modesto: l'architettura dueling dà il meglio in ambienti con molti stati in cui l'azione scelta cambia poco il risultato. Ciò che ti porti a casa da questo esercizio è lo scheletro completo per aggiungere, più avanti, gli altri cinque miglioramenti di Rainbow su questa stessa base.


## Punti chiave

- DQN, della lezione sette, è il punto di partenza; Rainbow è la somma di sei miglioramenti indipendenti che, combinati, superano nettamente DQN preso da solo
- I sei miglioramenti sono: doppio apprendimento-Q, replay dell'esperienza prioritizzato, architettura dueling, ritorni multi-passo, apprendimento distribuzionale (C51) e reti rumorose
- Lo studio di ablazione mostra che multi-passo, replay prioritizzato e apprendimento distribuzionale contribuiscono più di doppio apprendimento-Q e dueling, una volta combinati
- Rainbow raggiunge in circa sette milioni di fotogrammi ciò che DQN impiega circa 200 milioni a ottenere: circa 14 volte più efficienza sui dati
- L'apprendimento distribuzionale impara l'intera distribuzione dei ritorni possibili, non solo la loro media, e usa la divergenza KL come funzione di perdita
- RL multi-agente, RL gerarchico, meta-apprendimento/trasferimento/curriculum e sicurezza nell'RL sono linee di ricerca attive che vedrai citate in letteratura, anche se questa lezione non le sviluppa in profondità

## Prima della prossima lezione

**Apri il notebook di pratica di questa lezione e completa i due esercizi: l'architettura dueling da zero e i ritorni multi-passo.**

Nella lezione 11 cambi prospettiva: invece di migliorare un unico agente che impara da solo, vedrai architetture in cui più unità di calcolo si addestrano in parallelo, come A3C e IMPALA.
